# 03 — Feature Engineering

Runs `src/features/feature_engineering.create_features` and inspects the derived columns and a labeled example around a known failure.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import matplotlib.pyplot as plt
import pandas as pd

from src.config import load_config
from src.data.labels import create_labels
from src.data.validation import clean_sensor_data
from src.features.feature_engineering import create_features, feature_column_names

config = load_config()
sensors = clean_sensor_data(pd.read_parquet(config.paths.sensor_file))
failures = pd.read_csv(config.paths.failures_file, parse_dates=["failure_timestamp"])

fe = config.feature_engineering
features = create_features(
    sensors,
    sensor_columns=config.sensor_columns,
    rolling_window=fe.rolling_window,
    rolling_min_periods=fe.rolling_min_periods,
    zscore_window=fe.zscore_window,
    interpolation_limit=fe.interpolation_limit,
)
labeled = create_labels(features, failures, config.labeling["pre_failure_window_hours"])
labeled[feature_column_names(config.sensor_columns)].describe()

In [ ]:
example_failure = failures.iloc[0]
window = labeled[
    (labeled["machine_id"] == example_failure["machine_id"])
    & (labeled["timestamp"] >= example_failure["failure_timestamp"] - pd.Timedelta(hours=36))
    & (labeled["timestamp"] <= example_failure["failure_timestamp"])
].set_index("timestamp")
cols = ["vibration", "vibration_zscore", "temperature", "temperature_zscore", "anomaly"]
window[cols].plot(subplots=True, figsize=(10, 10))
plt.tight_layout()
plt.show()